In [0]:
from pyspark.sql.functions import (
    col,
    row_number
)

ZONE_PATH = "/Volumes/workspace/default/nyc_hvfhv_data/reference/taxi_zone_lookup.csv"

LOCATION_TABLE = "workspace.default.dim_location"

df_zones = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ZONE_PATH)
)

print("Taxi Zone Lookup loaded")
print("Columns:", df_zones.columns)

In [0]:
display(df_zones.limit(10))

In [0]:
# Step 12.3: Create dim_location

df_location = (
    df_zones
    .select(
        col("LocationID").cast("integer").alias("location_id"),
        col("Borough").alias("borough"),
        col("Zone").alias("zone"),
        col("service_zone").alias("service_zone")
    )
    .withColumn(
        "location_key",
        col("location_id")
    )
    .select(
        "location_key",
        "location_id",
        "borough",
        "zone",
        "service_zone"
    )
)

In [0]:
print("Dimension rows:", df_location.count())
print("Dimension columns:", len(df_location.columns))

display(
    df_location
    .orderBy("location_id")
    .limit(10)
)

In [0]:
display(
    df_location
    .groupBy("location_id")
    .count()
    .filter(col("count") > 1)
)

In [0]:
display(
    df_location
    .groupBy("location_key")
    .count()
    .filter(col("count") > 1)
)

In [0]:
(
    df_location
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(LOCATION_TABLE)
)

print("dim_location created successfully")
print("Table:", LOCATION_TABLE)